# Exploring drugbank data

In [3]:
from lxml import etree

In [17]:
# One complete drug structure in original XML format
ctx = etree.iterparse('drugbank_data.xml', events=('end',))
for _, el in ctx:
    if el.tag.endswith('}drug') and el.get('type'):
        print(etree.tostring(el, pretty_print=True).decode()[:4000])
        break

<drug xmlns="http://www.drugbank.ca" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" type="biotech" created="2005-06-13" updated="2026-02-02">
  <drugbank-id primary="true">DB00001</drugbank-id>
  <drugbank-id>BTD00024</drugbank-id>
  <drugbank-id>BIOD00024</drugbank-id>
  <name>Lepirudin</name>
  <description>Lepirudin is a recombinant hirudin formed by 65 amino acids that acts as a highly specific and direct thrombin inhibitor.[L41539,L41569] Natural hirudin is an endogenous anticoagulant found in _Hirudo medicinalis_ leeches.[L41539] Lepirudin is produced in yeast cells and is identical to natural hirudin except for the absence of sulfate on the tyrosine residue at position 63 and the substitution of leucine for isoleucine at position 1 (N-terminal end).[A246609]&#13;
&#13;
Lepirudin is used as an anticoagulant in patients with heparin-induced thrombocytopenia (HIT), an immune reaction associated with a high risk of thromboembolic complications.[A3, L41539] HIT is caused by th

In [4]:
NS = 'http://www.drugbank.ca'
TAG = lambda t: f'{{{NS}}}{t}'

In [6]:
tag_drug        = TAG('drug')
tag_drugbank    = TAG('drugbank')
tag_interaction = TAG('drug-interaction')
tag_target      = TAG('target')
tag_targets     = TAG('targets')
tag_pathway     = TAG('pathway')
tag_pathways    = TAG('pathways')
tag_enzyme      = TAG('enzyme')
tag_enzymes     = TAG('enzymes')

counts = {'drugs':0,'interactions':0,'targets':0,'pathways':0,'enzymes':0}
ctx = etree.iterparse('drugbank_data.xml', events=('end',), tag=tag_drug)


In [7]:
for _, el in ctx:
    parent = el.getparent()
    if parent is not None and parent.tag != tag_drugbank:
        el.clear()
        continue

    counts['drugs']        += 1
    counts['interactions'] += len(el.findall(f'.//{tag_interaction}'))
    counts['targets']      += len(el.findall(f'.//{tag_targets}/{tag_target}'))
    counts['pathways']     += len(el.findall(f'.//{tag_pathways}/{tag_pathway}'))
    counts['enzymes']      += len(el.findall(f'.//{tag_enzymes}/{tag_enzyme}'))

    el.clear()
    while el.getprevious() is not None:
        del el.getparent()[0]

    if counts['drugs'] % 500 == 0:
        print(counts)

print('FINAL:', counts)

{'drugs': 500, 'interactions': 398487, 'targets': 2571, 'pathways': 55896, 'enzymes': 1788}
{'drugs': 1000, 'interactions': 884806, 'targets': 4460, 'pathways': 56276, 'enzymes': 4160}
{'drugs': 1500, 'interactions': 1273100, 'targets': 6126, 'pathways': 57288, 'enzymes': 5527}
{'drugs': 2000, 'interactions': 1321091, 'targets': 7333, 'pathways': 57988, 'enzymes': 5658}
{'drugs': 2500, 'interactions': 1330772, 'targets': 8280, 'pathways': 129493, 'enzymes': 5682}
{'drugs': 3000, 'interactions': 1337422, 'targets': 9202, 'pathways': 129928, 'enzymes': 5716}
{'drugs': 3500, 'interactions': 1347582, 'targets': 10209, 'pathways': 130177, 'enzymes': 5731}
{'drugs': 4000, 'interactions': 1353205, 'targets': 11221, 'pathways': 158666, 'enzymes': 5772}
{'drugs': 4500, 'interactions': 1374009, 'targets': 12235, 'pathways': 205009, 'enzymes': 5887}
{'drugs': 5000, 'interactions': 1475518, 'targets': 12990, 'pathways': 205017, 'enzymes': 6160}
{'drugs': 5500, 'interactions': 1516509, 'targets': 1

## Testing connection with neo4j

In [9]:
from dotenv import load_dotenv
from neo4j import GraphDatabase
import os

load_dotenv()

URI      = os.getenv("NEO4J_URI")
USER     = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

print(f"URI:      {URI}")
print(f"User:     {USER}")
print(f"Database: {DATABASE}")

try:
    driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
    driver.verify_connectivity()
    print("\n Connection successful!")
except Exception as e:
    print(f"\n Connection failed: {e}")

URI:      neo4j+s://376f542a.databases.neo4j.io
User:     376f542a
Database: 376f542a

 Connection successful!


In [10]:
with driver.session(database=DATABASE) as session:
    result = session.run("""
        MATCH (n)
        RETURN labels(n)[0] AS label, count(n) AS count
        ORDER BY count DESC
    """)
    rows = result.data()

if rows:
    print("Existing nodes found:")
    for row in rows:
        print(f"  {row['label']}: {row['count']}")
else:
    print("Database is empty — ready for ingestion.")

Database is empty — ready for ingestion.


## Exploring data post ingestion
### counts overview

In [12]:
with driver.session(database=DATABASE) as s:
    result = s.run("""
        MATCH (n)
        RETURN labels(n)[0] AS label, count(n) AS count
        ORDER BY count DESC
    """)
    for row in result:
        print(f"  {row['label']:<20} {row['count']:>8,} nodes")

  Pathway                48,622 nodes
  Drug                    4,795 nodes
  ProteinTarget           3,272 nodes


In [13]:
with driver.session(database=DATABASE) as s:
    result = s.run("""
        MATCH ()-[r]->()
        RETURN type(r) AS relationship, count(r) AS count
        ORDER BY count DESC
    """)
    for row in result:
        print(f"  {row['relationship']:<25} {row['count']:>8,} edges")

  TARGETS                      2,012 edges
  METABOLIZED_BY               1,721 edges
  PART_OF                        653 edges
  INTERACTS_WITH                  12 edges


### Severity breakdown of interactions

In [14]:
with driver.session(database=DATABASE) as s:
    result = s.run("""
        MATCH ()-[r:INTERACTS_WITH]->()
        RETURN r.severity AS severity, count(r) AS count
        ORDER BY count DESC
    """)
    for row in result:
        print(f"  {row['severity']:<12} {row['count']:>8,}")

  Major              12


### Most connected drugs

In [15]:
with driver.session(database=DATABASE) as s:
    result = s.run("""
        MATCH (d:Drug)-[r:INTERACTS_WITH]-()
        RETURN d.name AS drug, count(r) AS interactions
        ORDER BY interactions DESC
        LIMIT 15
    """)
    for row in result:
        print(f"  {row['drug']:<35} {row['interactions']:>5} interactions")

  Valproate bismuth                       6 interactions
  Lasmiditan                              4 interactions
  Dofetilide                              1 interactions
  Hydroxyamphetamine                      1 interactions
  Amitriptylinoxide                       1 interactions
  Iofetamine I-123                        1 interactions
  Normethadone                            1 interactions
  Opium                                   1 interactions
  Piritramide                             1 interactions
  Torasemide                              1 interactions
  Citalopram                              1 interactions
  Dibenzepin                              1 interactions
  Quinupramine                            1 interactions
  Methoxyphenamine                        1 interactions
  Fluconazole                             1 interactions


### Most targetted proteins

In [16]:
with driver.session(database=DATABASE) as s:
    result = s.run("""
        MATCH (d:Drug)-[:TARGETS]->(t:ProteinTarget)
        RETURN t.name AS protein, t.gene AS gene, count(d) AS drug_count
        ORDER BY drug_count DESC
        LIMIT 15
    """)
    for row in result:
        print(f"  {row['gene']:<12} {row['protein']:<40} targeted by {row['drug_count']} drugs")

  HRH1         Histamine H1 receptor                    targeted by 26 drugs
  CHRM1        Muscarinic acetylcholine receptor M1     targeted by 24 drugs
  CHRM3        Muscarinic acetylcholine receptor M3     targeted by 23 drugs
  DRD2         D(2) dopamine receptor                   targeted by 21 drugs
  ESR1         Estrogen receptor                        targeted by 20 drugs
  CHRM2        Muscarinic acetylcholine receptor M2     targeted by 20 drugs
               DNA                                      targeted by 18 drugs
  PTGS2        Prostaglandin G/H synthase 2             targeted by 17 drugs
  CHRM4        Muscarinic acetylcholine receptor M4     targeted by 17 drugs
  AR           Androgen receptor                        targeted by 16 drugs
  HTR2A        5-hydroxytryptamine receptor 2A          targeted by 16 drugs
  NR1I2        Nuclear receptor subfamily 1 group I member 2 targeted by 16 drugs
  CHRM5        Muscarinic acetylcholine receptor M5     targeted by 16 